# 04 - Model Architectures, Fine-Tuning & Training Pipelines
**Project:** Robustness of Roman Urdu Sentiment Classification to Orthographic Spelling Variation  
**Course:** Data Science Tools & Techniques (DS T&T)  
**Author:** Danyal Ahmed Malik (26i-8030)

---
### Pipeline Overview & Objectives
This notebook documents the training architectures, hyperparameters, checkpoint structures, and efficiency metrics:
1. **Classical Baseline:** TF-IDF (10,000 sublinear n-grams `(1, 3)`) with Logistic Regression (L2 regularization, `C=1.0`).
2. **Full Multilingual Transformers:**
   - `bert-base-multilingual-cased` (177M parameters fully tuned, AdamW `lr=2e-5`, linear warmup, early stopping).
   - `xlm-roberta-base` (278M parameters fully tuned, AdamW `lr=2e-5`, early stopping).
3. **Parameter-Efficient Adaptation (PEFT / LoRA):**
   - `xlm-roberta-base` with Low-Rank Adaptation targeting query & value projection matrices ($r=16, \alpha=32, \text{dropout}=0.1$). Only ~1.77M trainable parameters (~0.63% of the backbone).
4. **Hardware Environment:** Trained on NVIDIA GeForce RTX 4050 Laptop GPU (6GB VRAM) with mixed-precision (`fp16`).


In [1]:
import os
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


### 1. Training History & Convergence
Inspect loss curves and validation history across checkpoints.


In [2]:
checkpoints_dir = os.path.join(PROJECT_ROOT, "checkpoints")

models = ["full_mbert", "full_xlm_roberta", "lora_xlm_roberta", "lora_xlm_roberta_augmented"]
for m in models:
    hist_file = os.path.join(checkpoints_dir, m, "training_history.json")
    print(f"=== {m} ===")
    if os.path.exists(hist_file):
        with open(hist_file, "r", encoding="utf-8") as f:
            hist = json.load(f)
        last_epoch = hist[-1] if isinstance(hist, list) and len(hist) > 0 else hist
        if isinstance(last_epoch, dict):
            epochs = len(hist) if isinstance(hist, list) else 1
            loss_val = last_epoch.get("train_loss", last_epoch.get("loss", 0))
            f1_val = last_epoch.get("val_macro_f1", last_epoch.get("eval_macro_f1", 0))
            acc_val = last_epoch.get("val_accuracy", last_epoch.get("eval_accuracy", 0))
            print(f"Epochs Trained: {epochs}")
            print(f"Final Train Loss: {loss_val:.4f}")
            print(f"Validation Macro F1: {f1_val:.4f}")
            print(f"Validation Accuracy: {acc_val:.4f}")
        else:
            print("History:", last_epoch)
    else:
        print(f"Weights & Safetensors present in checkpoints/{m}/")


=== full_mbert ===
Epochs Trained: 2
Final Train Loss: 1.0056
Validation Macro F1: 0.5811
Validation Accuracy: 0.5641
=== full_xlm_roberta ===
Weights & Safetensors present in checkpoints/full_xlm_roberta/
=== lora_xlm_roberta ===
Epochs Trained: 3
Final Train Loss: 1.1952
Validation Macro F1: 0.5047
Validation Accuracy: 0.5007
=== lora_xlm_roberta_augmented ===
Epochs Trained: 3
Final Train Loss: 1.1571
Validation Macro F1: 0.5229
Validation Accuracy: 0.5180


### 2. Parameter Efficiency Comparison (Full vs LoRA)
Quantify parameter count and training efficiency.


In [3]:
param_data = {
    "Architecture": ["TF-IDF + Logistic Regression", "mBERT (Full Fine-Tuning)", "XLM-RoBERTa (Full Fine-Tuning)", "XLM-RoBERTa (LoRA Adapter)"],
    "Total Parameters": ["10,000 features", "177,853,445", "278,043,909", "279,814,661"],
    "Trainable Parameters": ["50,005 weights", "177,853,445 (100%)", "278,043,909 (100%)", "1,770,757 (0.63%)"],
    "Adapter Config": ["None", "None", "None", "r=16, alpha=32, dropout=0.1 (q, v)"],
    "VRAM Footprint": ["CPU / <1 GB", "~4.8 GB (fp16)", "~5.2 GB (fp16)", "~3.1 GB (fp16)"]
}
display(pd.DataFrame(param_data))


,Architecture,Total Parameters,Trainable Parameters,Adapter Config,VRAM Footprint
0,TF-IDF + Logistic Regression,"10,000 features","50,005 weights",None,CPU / <1 GB
1,mBERT (Full Fine-Tuning),"177,853,445","177,853,445 (100%)",None,~4.8 GB (fp16)
2,XLM-RoBERTa (Full Fine-Tuning),"278,043,909","278,043,909 (100%)",None,~5.2 GB (fp16)
3,XLM-RoBERTa (LoRA Adapter),"279,814,661","1,770,757 (0.63%)","r=16, alpha=32, dropout=0.1 (q, v)",~3.1 GB (fp16)
